# The Barren Plateau Challenge: Navigating the Training Landscape

**QOSF Monthly Challenge - [Jun 2025]**

**Author:** Tan Jun Liang

---

## 1. Introduction: The Silent Obstacle in QML

Variational Quantum Algorithms (VQAs) are a promising class of algorithms for near-term quantum computers. They work by using a classical optimizer to train the parameters of a quantum circuit. However, as the number of qubits and the depth of these circuits grow, we encounter a significant obstacle: **Barren Plateaus**.

> A barren plateau is a region in the landscape of a cost function where the gradient is, on average, exponentially close to zero.

When the gradient vanishes, the classical optimizer has no information about which direction to move the parameters to improve the solution. This causes the training to stagnate.

**The Goal of This Challenge:** You will first witness the barren plateau effect. Then, your challenge is to implement and compare different modern techniques to mitigate it and successfully train a Parameterized Quantum Circuit (PQC).

In [ ]:
# --- General Imports ---
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm # For progress bars

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient
from qiskit_algorithms.optimizers import ADAM

# --- Matplotlib settings for prettier plots ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (10, 6)})

### A Quick Note on Qiskit Primitives

This notebook uses Qiskit's modern `primitives` API. Primitives are fundamental building blocks for quantum algorithms. Here are the two we'll be using:

*   **`Estimator`**: This primitive runs a quantum circuit and calculates the expectation value of an observable (like a Pauli operator). It's the core tool for determining our circuit's output.
*   **`ParamShiftEstimatorGradient`**: This primitive takes an `Estimator` and uses it to automatically calculate the gradient of the circuit's parameters. This is what our classical optimizer will use to find the right direction for updates.

We provide these so you can focus on the high-level logic of the mitigation techniques, not the low-level gradient implementation.

## 2. The Baseline: A Known Barren Plateau

We'll start by setting up a problem that we know suffers from a barren plateau.

*   **The Task:** A simple regression problem. We will train a PQC to learn `f(x) = np.sin(np.pi * x)` for `x` in `[-1, 1]`.
*   **The Ansatz:** We will use a Hardware-Efficient Ansatz, which is known to exhibit barren plateaus. It consists of layers of `Ry` rotations and `CZ` entangling gates.
*   **The Cost Function:** Our cost will be the Mean Squared Error (MSE). The PQC's output is the expectation value of a **local observable**: the Pauli Z operator on the first qubit, `<Z₀>`.

In [ ]:
def build_pqc(num_qubits, depth):
    """Builds the hardware-efficient PQC."""
    qc = QuantumCircuit(num_qubits)
    params = []
    # Use a different Parameter object for each rotation
    for d in range(depth):
        for i in range(num_qubits):
            # Create a unique parameter for each gate
            param = Parameter(f'p_{d}_{i}')
            params.append(param)
            qc.ry(param, i)
        if d < depth - 1: # Add entangling layer after each rotation layer except the last one
            for i in range(num_qubits - 1):
                qc.cz(i, i + 1)
            # Optional: add entanglement between the last and first qubits
            if num_qubits > 1:
                qc.cz(num_qubits - 1, 0)
    return qc, params

# We instantiate the primitives here
estimator = Estimator()
gradient = ParamShiftEstimatorGradient(estimator)

### 2.1 Visualizing the Vanishing Gradient

Before we train, let's prove the barren plateau exists. We calculate the *variance* of the gradient for a single parameter, averaged over many random initializations. As the plot below shows, this variance decays exponentially as we add more qubits.

*(This plot was pre-computed to save you time. The code to generate it is in the collapsed cell below for your reference, but **you do not need to run it**.)*

![Pre-computed Barren Plateau Plot](https://i.imgur.com/uNfQ0tC.png)

In [ ]:
# --- WARNING: THIS CELL IS SLOW AND FOR REFERENCE ONLY ---

# qubit_counts = [4, 6, 8, 10]
# variances = []
# n_trials = 50 # Number of random initializations

# for n_qubits in qubit_counts:
#     pqc, params = build_pqc(n_qubits, depth=n_qubits)
#     local_observable = SparsePauliOp("Z" + "I" * (n_qubits - 1))
#     grads = []
#     for _ in range(n_trials):
#         rand_params = np.random.uniform(0, 2 * np.pi, len(params))
#         grad_result = gradient.run(pqc, local_observable, [rand_params]).result().gradients[0][0]
#         grads.append(grad_result)
#     variances.append(np.var(grads))

# # --- Plotting the result ---
# plt.plot(qubit_counts, variances, 'o-', label='Gradient Variance')
# plt.yscale('log')
# plt.xlabel('Number of Qubits')
# plt.ylabel('Gradient Variance (log scale)')
# plt.title('Demonstration of the Barren Plateau')
# plt.legend()
# plt.show()

In [ ]:
# --- Let's fix our problem size ---
NUM_QUBITS = 8
DEPTH = 8

# --- Define the training data ---
X_train = np.linspace(-1, 1, 50)
Y_train = np.sin(np.pi * X_train)

# --- The PQC for our problem ---
pqc, params = build_pqc(NUM_QUBITS, DEPTH)
observable = SparsePauliOp("Z" + "I" * (NUM_QUBITS - 1))

def objective_function(p_values):
    """Calculates the expectation value for a given set of parameters."""
    job = estimator.run(pqc, observable, [p_values])
    est_val = job.result().values[0]
    return est_val

def cost_function(p_values):
    """Calculates the MSE cost."""
    # We will "encode" our classical data x by rotating the first qubit
    # For simplicity, we are not training this encoding, just using it to generate our dataset.
    # We will train the PQC to learn the mapping from a fixed initial state to the target sin wave.
    # This is a common pattern in QML for function fitting.
    
    predictions = [objective_function(np.concatenate(([x], p_values[1:]))) for x in X_train]
    cost = np.mean((predictions - Y_train)**2)
    return cost
    
def training_loop(initial_params, cost_func, optimizer, max_iter=100):
    """A generic training loop."""
    cost_history = []
    params_history = [initial_params]

    p_values = initial_params
    for i in tqdm(range(max_iter)):
        # Calculate gradient
        grad_vals = optimizer.grad_function(p_values)
        
        # Update parameters
        p_values = optimizer.update(p_values, grad_vals)

        # Calculate and store cost
        cost = cost_func(p_values)
        cost_history.append(cost)
        params_history.append(p_values)
    
    return cost_history, params_history


# --- Baseline Training ---
print("Running Baseline Training (Random Initialization)...")
optimizer = ADAM(maxiter=100, lr=0.01)
optimizer.grad_function = gradient.run # Hook up the gradient primitive
initial_params_baseline = np.random.uniform(0, 2 * np.pi, len(params))
baseline_cost, _ = training_loop(initial_params_baseline, cost_function, optimizer, max_iter=100)

# --- Plotting ---
plt.plot(baseline_cost, label='Baseline (Random Init)')
plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Baseline Training Attempt')
plt.legend()
plt.show()

## 3. Your Challenge: Escaping the Plateau

The baseline training failed, as the cost stagnated. Your task is to implement and compare mitigation strategies.

### Task 1: Mitigation via Parameter Initialization

**Theory:** A key cause of barren plateaus in deep circuits is initializing parameters across the full `[0, 2π]` space. This often creates a circuit that behaves like a random unitary transformation, which scrambles information and leads to vanishing gradients. A simple fix is to initialize all parameters from a narrow distribution around zero. This keeps the initial circuit close to the identity and helps preserve the gradient signal.

**Your Task:** Copy the baseline training code and modify it to initialize all parameters from a narrow uniform distribution, for example, `np.random.uniform(0, 0.01)`.

In [ ]:
# YOUR CODE HERE
# 1. Define a new set of initial parameters with narrow initialization.
# 2. Run the training_loop with these new parameters.
# 3. Plot the resulting cost history.

print("Running Training with Narrow Initialization...")
# Hint: Change the initialization range from [0, 2*pi] to something smaller.
initial_params_narrow = np.random.uniform(0, 0.01, len(params)) # <- MODIFY THIS
narrow_cost, _ = training_loop(initial_params_narrow, cost_function, optimizer, max_iter=100)

plt.plot(narrow_cost, label='Narrow Init', color='green')
plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Training with Narrow Parameter Initialization')
plt.legend()
plt.show()

### Task 2: Mitigation via Cost Function Choice

**Theory:** Another major cause of barren plateaus is the choice of cost function. A **global cost function**, which depends on measurements across many or all qubits (e.g., `<Z₀⊗Z₁⊗...⊗Zₙ>`), almost always has barren plateaus. In contrast, a **local cost function**, which only depends on a few qubits (like our `<Z₀>`), can avoid this specific cause.

**Your Task:**
1.  Define a **global observable** (e.g., `SparsePauliOp("Z" * NUM_QUBITS)`).
2.  Create a new `cost_function_global` that uses this new observable.
3.  Run the training loop using this global cost function (you can use either random or narrow initialization).
4.  Compare the performance to the local cost function. You should see that even with narrow initialization, the global cost function fails to train.

In [ ]:
## YOUR CODE HERE
# 1. Define a global observable.
global_observable = SparsePauliOp("Z" * NUM_QUBITS)

# 2. Define a new objective and cost function for the global case.
def objective_function_global(p_values):
    job = estimator.run(pqc, global_observable, [p_values]) # Use global_observable
    return job.result().values[0]
def cost_function_global(p_values):
    predictions = [objective_function_global(np.concatenate(([x], p_values[1:]))) for x in X_train]
    return np.mean((predictions - Y_train)**2)

## 3. Run the training with the global cost function.
print("Running Training with Global Cost Function (Narrow Init)...")

# Let's use the better narrow initialization to give it a fair chance.
global_cost, _ = training_loop(initial_params_narrow, cost_function_global, optimizer, max_iter=100)
plt.plot(global_cost, label='Global Cost', color='red')
plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Training with a Global Cost Function')
plt.legend()
plt.show()


#### **Cell 13: Bonus Challenge (Markdown)**
### Bonus Challenge (Advanced): Layer-by-Layer Training

For those looking for a tougher challenge, implement a layer-by-layer training scheme.
1.  Start with a PQC of `depth=1`. Train its parameters until convergence.
2.  *Freeze* the parameters of the first layer.
3.  Add a second layer to the PQC.
4.  Train *only* the parameters of the new, second layer.
5.  Repeat until you reach the desired total depth.

This is more complex to implement but can be very effective. You will need to carefully manage which parameters are trainable at each step.

## 4. Analysis and Comparison

Now, let's compare the performance of all the methods you've tried. A fair comparison uses a consistent metric. For this challenge, we will compare the **final cost (MSE) achieved after a fixed number of optimizer steps (100)**.

**Your Task:** Create a single plot that shows the cost function versus training epochs for:
1.  The original (failing) baseline.
2.  The narrow initialization strategy (with the local cost function).
3.  The global cost function strategy.

In [ ]:
# YOUR CODE HERE
# Generate the final comparison plot showing all loss histories on one graph.
plt.plot(baseline_cost, label='Baseline (Local Cost, Random Init)')
plt.plot(narrow_cost, label='Mitigated (Local Cost, Narrow Init)', color='green')
plt.plot(global_cost, label='Failed (Global Cost, Narrow Init)', color='red')

plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Comparison of Training Strategies')
plt.legend(loc='best')
plt.show()

## 5. Written Analysis

In the markdown cell below, please describe your findings.
*   Which mitigation strategy performed the best and why?
*   Why did the global cost function fail to train, even with good initialization?
*   What are the trade-offs for each method (e.g., implementation complexity)?
*   Did you try the bonus challenge or any other creative ideas? If so, what were they and how did they perform?

---

*... Your analysis here ...*

## 6. Bonus Ideas & Going Further

Want to stand out? Here are some other ideas to explore:
*   **Different Optimizers:** How does `ADAM` compare to a gradient-free optimizer like `SPSA` in the presence of barren plateaus?
*   **Ansatz Architecture:** Does changing the entangling gate from `CZ` to `CX` affect the results? What about changing the connectivity?
*   **Combining Techniques:** Can you combine layer-by-layer training with narrow initialization for even better results?

## 7. How to Submit

Please submit this completed Jupyter Notebook. Ensure that all the plots are visible and that your written analysis is complete. Good luck!